In [ ]:
import os
import re
import sys
import json
import difflib
import numpy as np
import pandas as pd
from pathlib import Path
from tqdm.auto import tqdm

# ── 项目根目录解析 ──
PROJECT_ROOT = Path(os.path.abspath("")).resolve()
while not (PROJECT_ROOT / "configs").exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
print(f"Project root: {PROJECT_ROOT}")

from src.utils.cwe import CWEUtils
from src.utils.cve import CVEUtils
from src.utils.formatter import auto_format_c_code_batch, ClangFormatStyle

# ── 超参数 ──
MIN_CODE_LEN         = 50       # 最短代码字符数（过滤过短的无意义代码）
MAX_CODE_LEN         = 65535    # 最长代码字符数（Milvus VARCHAR 上限）
MERGE_FUNCS_AS_CVE   = True    # 是否按 (cve_id, commit_id, cwe_id) 合并同源多函数
DROP_OUTDATED        = True     # 是否丢弃论文中标注的 outdated 补丁（outdated == 1）
DROP_NVD_PLACEHOLDER = True     # 是否过滤 NVD-CWE-noinfo / NVD-CWE-Other 占位符
KEEP_ALL_CWES        = True     # True: 多 CWE 拆分为多行；False: 仅取首个真实 CWE-xxx

# ── 数据路径 ──
DATASET_DIR  = PROJECT_ROOT / "src/preprocessing/vul/ReposVul/dataset/processed"
SOURCE_FILES = [
    DATASET_DIR / "train_c_cpp_repository.jsonl",
    DATASET_DIR / "valid_c_cpp_repository.jsonl",
    DATASET_DIR / "test_c_cpp_repository.jsonl",
]

if MERGE_FUNCS_AS_CVE:
    OUTPUT_PATH = PROJECT_ROOT / "data/raw/vul/ReposVul_Merged_v2.jsonl"
else:
    OUTPUT_PATH = PROJECT_ROOT / "data/raw/vul/ReposVul_v2.jsonl"

print(f"Output path:  {OUTPUT_PATH}")
print(f"超参数:")
print(f"  MIN_CODE_LEN         = {MIN_CODE_LEN}")
print(f"  MAX_CODE_LEN         = {MAX_CODE_LEN}")
print(f"  MERGE_FUNCS_AS_CVE   = {MERGE_FUNCS_AS_CVE}")
print(f"  DROP_OUTDATED        = {DROP_OUTDATED}")
print(f"  DROP_NVD_PLACEHOLDER = {DROP_NVD_PLACEHOLDER}")
print(f"  KEEP_ALL_CWES        = {KEEP_ALL_CWES}")

Project root: <PROJECT_ROOT>
Output path:  <PROJECT_ROOT>/data/raw/vul/ReposVul_Merged.jsonl
超参数:
  MIN_CODE_LEN         = 35
  MAX_CODE_LEN         = 65535
  MERGE_FUNCS_AS_CVE   = True
  DROP_OUTDATED        = True
  DROP_NVD_PLACEHOLDER = True
  KEEP_ALL_CWES        = True


<CONDA_ENV>/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# ── 流式加载 + 即时过滤原始数据 ──
# 三个 processed jsonl 合计约 3GB，必须在迭代过程中尽早过滤掉
# target == 0 的负样本（占 95%+），以及 outdated 与不合法的样本，
# 否则 DataFrame 会膨胀到数十 GB。

KEEP_FIELDS = (
    "function",         # → vul_code
    "cve_id",
    "cwe_id",           # list
    "cve_language",     # → language
    "cve_description",  # → cve_desc_raw
    "cwe_description",  # → cwe_desc_raw
    "function_numbers", # dict: {function_start, function_end}
    "line_numbers",     # list of {line_change, line_start, line_end}
    "commit_id",
    "outdated",
)


def _quick_valid(val) -> bool:
    """轻量级字符串非空判断。"""
    if not isinstance(val, str):
        return False
    s = val.strip()
    return len(s) > 0 and s.lower() != "none"


def _cwe_list_valid(cwe_field) -> bool:
    """cwe_id 必须是非空 list 且至少含一个非空字符串。"""
    if not isinstance(cwe_field, list) or len(cwe_field) == 0:
        return False
    return any(_quick_valid(c) for c in cwe_field)


records = []
per_file_total = {}
per_file_kept = {}

for src_path in SOURCE_FILES:
    name = src_path.stem.split("_")[0]
    total = 0
    kept = 0
    with open(src_path, "r", encoding="utf-8") as f:
        for line in tqdm(f, desc=f"流式过滤 {name}"):
            total += 1
            try:
                record = json.loads(line)
            except json.JSONDecodeError:
                continue

            # 流式过滤①：仅保留 target == 1（漏洞函数行）
            if record.get("target") != 1:
                continue

            # 流式过滤②：丢弃过时补丁
            if DROP_OUTDATED and record.get("outdated") == 1:
                continue

            # 流式过滤③：仅保留 C / C++
            lang = record.get("cve_language")
            if lang not in ("C", "C++"):
                continue

            # 流式过滤④：function 非空且长度合法
            func = record.get("function")
            if not _quick_valid(func):
                continue
            if not (MIN_CODE_LEN <= len(func) <= MAX_CODE_LEN):
                continue

            # 流式过滤⑤：CVE / CWE 标识必须非空
            if not _quick_valid(record.get("cve_id")):
                continue
            if not _cwe_list_valid(record.get("cwe_id")):
                continue

            # 流式过滤⑥：line_numbers 必须存在，否则无法重建 fixed_code
            if not record.get("line_numbers"):
                continue

            # 流式过滤⑦：function_numbers 中的 start/end 必须可解析
            fn = record.get("function_numbers") or {}
            if not isinstance(fn, dict):
                continue
            if fn.get("function_start") is None or fn.get("function_end") is None:
                continue

            records.append({k: record.get(k) for k in KEEP_FIELDS})
            kept += 1

    per_file_total[name] = total
    per_file_kept[name] = kept
    print(f"  {name}: 原始 {total} 行 → 过滤后保留 {kept} 行")

df_raw = pd.DataFrame(records)
del records

print()
print(f"全部原始行数:     {sum(per_file_total.values()):>7}")
print(f"流式过滤后样本数: {len(df_raw):>7}")
df_raw[["cve_id", "cwe_id", "cve_language", "outdated", "commit_id"]].head()

流式过滤 train: 185791it [01:30, 2063.06it/s]


  train: 原始 185791 行 → 过滤后保留 4573 行


流式过滤 valid: 23224it [00:14, 1651.26it/s]


  valid: 原始 23224 行 → 过滤后保留 593 行


流式过滤 test: 23224it [00:13, 1695.02it/s]

  test: 原始 23224 行 → 过滤后保留 556 行

全部原始行数:      232239
流式过滤后样本数:    5722


,cve_id,cwe_id,cve_language,outdated,commit_id
0,CVE-2022-24754,"[CWE-787, CWE-120, CWE-1284]",C,0,d27f79da11df7bc8bb56c2f291d71e54df8d2c47
1,CVE-2012-3552,[CWE-362],C,0,f6d8bd051c391c1c0458a30b2a7abcd939329259
2,CVE-2023-23457,[CWE-119],C++,0,779b648c5f6aa9b33f4728f79dd4d0efec0bf860
3,CVE-2012-2319,[CWE-264],C,0,6f24f892871acc47b40dd594c63606a17c714f77
4,CVE-2020-29260,[CWE-400],C,0,bef41f6ec4097a8ee094f90a1b34a708fbd757ec


In [3]:
# ── Cell 2: 通过 unified diff 重建 fixed_code ──
#
# ReposVul 的 processed jsonl 不直接提供修复后的函数，只给出：
#   - `function`:         漏洞函数源码（即 vul_code，函数级别）
#   - `function_numbers`: {function_start, function_end} 函数在文件中的绝对行号
#   - `line_numbers`:     [{line_change, line_start, line_end}, ...] 关联的 unified diff hunks
#
# 思路：
#   1. 解析 hunk 头部 "@@ -A,B +C,D @@" 提取 before/after 长度；
#   2. 把 hunk body 拆为「before 片段」(' '/'-' 行) 与「after 片段」(' '/'+' 行)；
#   3. 用 `line_start - function_start` 把 hunk 的绝对行号转换到函数内的相对行号；
#   4. 优先按相对行号定位锚点；若锚点与 before 片段不完全一致，再回退到
#      在 vul_code 中查找 before 片段唯一子串并替换；
#   5. 若 hunk 的纯变更行（+/-）全部落在函数行范围之外（仅 context 与函数重叠），
#      判定为「diff 不属于本函数」，整条样本返回 None 由后续 cell 丢弃。

HUNK_HDR_RE = re.compile(r"^@@\s*-(\d+)(?:,(\d+))?\s*\+(\d+)(?:,(\d+))?\s*@@")


def _parse_hunk(line_change: str):
    """解析单个 unified diff hunk。

    返回 dict::

        {
            "before_start": int,   # 原文件起始行号 (1-based)
            "before_len":   int,
            "after_start":  int,
            "after_len":    int,
            "before_lines": [str], # ' '/'-' 行（去掉首字符）
            "after_lines":  [str], # ' '/'+' 行（去掉首字符）
            "minus_count":  int,   # '-' 行数
            "plus_count":   int,   # '+' 行数
        }

    解析失败时返回 ``None``。
    """
    if not isinstance(line_change, str) or not line_change:
        return None

    lines = line_change.splitlines()
    if not lines:
        return None

    # 找到 hunk 头部行（ReposVul 中部分 hunk 头部前后可能多一个空格）
    header_idx = None
    for i, ln in enumerate(lines):
        if ln.lstrip().startswith("@@"):
            header_idx = i
            break
    if header_idx is None:
        return None

    m = HUNK_HDR_RE.match(lines[header_idx].lstrip())
    if not m:
        return None

    before_start = int(m.group(1))
    before_len   = int(m.group(2)) if m.group(2) is not None else 1
    after_start  = int(m.group(3))
    after_len    = int(m.group(4)) if m.group(4) is not None else 1

    before_lines = []
    after_lines = []
    minus_count = 0
    plus_count = 0
    for body in lines[header_idx + 1:]:
        if not body:
            # 空字符串视为空 context 行（diff 输出尾随空行）
            before_lines.append("")
            after_lines.append("")
            continue
        prefix = body[0]
        content = body[1:]
        if prefix == "-":
            before_lines.append(content)
            minus_count += 1
        elif prefix == "+":
            after_lines.append(content)
            plus_count += 1
        elif prefix == " ":
            before_lines.append(content)
            after_lines.append(content)
        elif prefix == "\\":
            # "\\ No newline at end of file" 之类的注释行，忽略
            continue
        else:
            # 极少数情况下首字符是 tab 或其他控制符，按 context 处理
            before_lines.append(body)
            after_lines.append(body)

    return {
        "before_start": before_start,
        "before_len":   before_len,
        "after_start":  after_start,
        "after_len":    after_len,
        "before_lines": before_lines,
        "after_lines":  after_lines,
        "minus_count":  minus_count,
        "plus_count":   plus_count,
    }


def _any_change_in_range(hunk: dict, fn_start: int, fn_end: int) -> bool:
    """判断 hunk 的变更区间是否与函数行号区间存在重叠。

    若 hunk 的 [before_start, before_start+before_len-1] 与函数范围相交，
    且 hunk 内至少含有一个 '-' 或 '+' 行，则认为变更可能落在函数内。
    """
    if hunk["minus_count"] == 0 and hunk["plus_count"] == 0:
        return False
    before_end = hunk["before_start"] + max(hunk["before_len"], 1) - 1
    return not (before_end < fn_start or hunk["before_start"] > fn_end)


def _apply_hunk_by_lineno(func_lines: list, hunk: dict, fn_start: int):
    """优先策略：基于绝对行号在函数内定位 before 片段并替换为 after 片段。

    返回 (new_func_lines, ok)，ok=False 时调用方需要回退到字符串查找。
    """
    rel = hunk["before_start"] - fn_start  # 0-based 相对行号
    if rel < 0:
        return func_lines, False
    end = rel + len(hunk["before_lines"])
    if end > len(func_lines):
        return func_lines, False

    window = func_lines[rel:end]
    if window == hunk["before_lines"]:
        return func_lines[:rel] + list(hunk["after_lines"]) + func_lines[end:], True
    return func_lines, False


def _apply_hunk_by_substring(func_lines: list, hunk: dict):
    """回退策略：在 func_lines 中查找 before_lines 子序列（必须唯一）并替换为 after_lines。

    返回 (new_func_lines, ok)。仅在唯一匹配时执行替换。
    """
    before = hunk["before_lines"]
    n = len(before)
    if n == 0:
        return func_lines, False

    matches = []
    for i in range(0, len(func_lines) - n + 1):
        if func_lines[i:i + n] == before:
            matches.append(i)
            if len(matches) > 1:
                return func_lines, False  # 不唯一 → 放弃

    if len(matches) != 1:
        return func_lines, False

    i = matches[0]
    return func_lines[:i] + list(hunk["after_lines"]) + func_lines[i + n:], True


def apply_hunks_to_function(vul_code: str, function_numbers: dict, line_numbers: list):
    """将一组 unified diff hunks 应用到函数级 vul_code 上，重建 fixed_code。

    返回 ``str``（成功）或 ``None``（diff 与函数不匹配，应丢弃）。
    """
    if not vul_code or not function_numbers or not line_numbers:
        return None

    fn_start = function_numbers.get("function_start")
    fn_end = function_numbers.get("function_end")
    if fn_start is None or fn_end is None:
        return None

    # 解析所有 hunks
    parsed_hunks = []
    for hk in line_numbers:
        if not isinstance(hk, dict):
            continue
        p = _parse_hunk(hk.get("line_change", ""))
        if p is not None:
            parsed_hunks.append(p)
    if not parsed_hunks:
        return None

    # 过滤掉与本函数完全无关的 hunk（变更行不与函数范围重叠）
    relevant = [h for h in parsed_hunks if _any_change_in_range(h, fn_start, fn_end)]
    if not relevant:
        # 所有 hunk 的变更都在别的函数 → 丢弃该样本
        return None

    # 保留换行格式：splitlines(keepends=False) + 显式记录尾部换行
    has_trailing_newline = vul_code.endswith("\n")
    func_lines = vul_code.split("\n")
    if has_trailing_newline and func_lines and func_lines[-1] == "":
        # 末尾空字符串是 trailing newline 产物，避免影响匹配
        func_lines.pop()

    # 按 before_start 从大到小应用，避免前置变更影响后续行号
    relevant_sorted = sorted(relevant, key=lambda h: h["before_start"], reverse=True)

    any_applied = False
    for hk in relevant_sorted:
        new_lines, ok = _apply_hunk_by_lineno(func_lines, hk, fn_start)
        if not ok:
            new_lines, ok = _apply_hunk_by_substring(func_lines, hk)
        if ok:
            func_lines = new_lines
            any_applied = True
        # 若某 hunk 应用失败：跳过它（保留其他成功 hunk 的修改）
        # 这种情况通常是 hunk 同时影响多个相邻函数，未匹配上的部分本就属于别的函数

    if not any_applied:
        return None

    fixed_code = "\n".join(func_lines)
    if has_trailing_newline:
        fixed_code += "\n"
    return fixed_code


# ── 应用到 DataFrame ──
df = df_raw.copy()
print(f"开始重建 fixed_code, 输入样本数: {len(df)}")

fixed_codes = []
for _, row in tqdm(df.iterrows(), total=len(df), desc="应用 unified diff"):
    fixed_codes.append(
        apply_hunks_to_function(row["function"], row["function_numbers"], row["line_numbers"])
    )

df["vul_code"] = df["function"]
df["fixed_code"] = fixed_codes

before_drop = len(df)
df = df[df["fixed_code"].notna()].reset_index(drop=True)
dropped = before_drop - len(df)
print(f"diff 重建失败被丢弃: {dropped:>5} 条")
print(f"diff 重建成功:       {len(df):>5} 条")

df[["cve_id", "vul_code", "fixed_code"]].head(1)

开始重建 fixed_code, 输入样本数: 5722


应用 unified diff: 100%|██████████| 5722/5722 [00:00<00:00, 16452.51it/s]


diff 重建失败被丢弃:  1938 条
diff 重建成功:        3784 条


,cve_id,vul_code,fixed_code
0,CVE-2022-24754,static pj_status_t respond_digest( pj_pool_t *...,static pj_status_t respond_digest( pj_pool_t *...


In [4]:
# ── Cell 3: 重建后的清洗 ──
# 1) vul_code / fixed_code 非空字符串
# 2) vul_code != fixed_code（diff 应用后必须真的产生修改）
# 3) 两者长度 ∈ [MIN_CODE_LEN, MAX_CODE_LEN]

def _is_valid_str(val) -> bool:
    if val is None or not isinstance(val, str):
        return False
    s = val.strip()
    return len(s) > 0 and s.lower() != "none"


before = len(df)
df = df[df["vul_code"].apply(_is_valid_str) & df["fixed_code"].apply(_is_valid_str)]
print(f"步骤1 (代码非空):          {len(df):>6} 条 (移除 {before - len(df)})")

before = len(df)
df = df[df["vul_code"] != df["fixed_code"]]
print(f"步骤2 (vul != fixed):       {len(df):>6} 条 (移除 {before - len(df)})")

before = len(df)
vul_len = df["vul_code"].str.len()
fix_len = df["fixed_code"].str.len()
df = df[
    vul_len.between(MIN_CODE_LEN, MAX_CODE_LEN, inclusive="both")
    & fix_len.between(MIN_CODE_LEN, MAX_CODE_LEN, inclusive="both")
]
print(f"步骤3 (代码长度合法):       {len(df):>6} 条 (移除 {before - len(df)})")

df = df.reset_index(drop=True)
print(f"\n清洗后总样本数: {len(df)}")
df[["cve_id", "cwe_id", "cve_language", "vul_code", "fixed_code"]].head(2)

步骤1 (代码非空):            3784 条 (移除 0)
步骤2 (vul != fixed):         3784 条 (移除 0)
步骤3 (代码长度合法):         3784 条 (移除 0)

清洗后总样本数: 3784


,cve_id,cwe_id,cve_language,vul_code,fixed_code
0,CVE-2022-24754,"[CWE-787, CWE-120, CWE-1284]",C,static pj_status_t respond_digest( pj_pool_t *...,static pj_status_t respond_digest( pj_pool_t *...
1,CVE-2023-23457,[CWE-119],C++,void\nPackLinuxElf64::PackLinuxElf64help1(Inpu...,void\nPackLinuxElf64::PackLinuxElf64help1(Inpu...


In [5]:
# ── Cell 4: 多 CWE 展开 + NVD 占位符过滤 ──
#
# ReposVul 的 cwe_id 是列表（一条样本可关联多个 CWE）。同时含有
# 大量 "NVD-CWE-noinfo" / "NVD-CWE-Other" 占位符，这些不是真实 CWE。
#
# 行为：
#   - KEEP_ALL_CWES=True:  对每条样本按 cwe_id 列表 explode；先剔除占位符，
#                          剩余 CWE 各占一行；如果剔除后该样本无任何真实 CWE，
#                          整条丢弃（受 DROP_NVD_PLACEHOLDER 控制）。
#   - KEEP_ALL_CWES=False: 仅保留首个真实 CWE-xxx，作为标量字段。

NVD_PLACEHOLDERS = {"NVD-CWE-noinfo", "NVD-CWE-Other"}


def _clean_cwe_list(cwe_list):
    """过滤掉占位符与无效值，返回去重后的真实 CWE-xxx 列表（保持原顺序）。"""
    if not isinstance(cwe_list, list):
        return []
    seen = set()
    out = []
    for c in cwe_list:
        if not isinstance(c, str):
            continue
        cs = c.strip()
        if not cs:
            continue
        if DROP_NVD_PLACEHOLDER and cs in NVD_PLACEHOLDERS:
            continue
        if cs in seen:
            continue
        seen.add(cs)
        out.append(cs)
    return out


df["cwe_id"] = df["cwe_id"].apply(_clean_cwe_list)

before = len(df)
df = df[df["cwe_id"].apply(lambda lst: len(lst) > 0)].reset_index(drop=True)
print(f"过滤无真实 CWE 的样本:   {len(df):>6} 条 (移除 {before - len(df)})")

if KEEP_ALL_CWES:
    df = df.explode("cwe_id").reset_index(drop=True)
    print(f"按 CWE explode 后:        {len(df):>6} 条")
else:
    df["cwe_id"] = df["cwe_id"].apply(lambda lst: lst[0])
    print(f"仅取首个真实 CWE:         {len(df):>6} 条")

# 重命名为后续 cell 习惯使用的字段名
df = df.rename(columns={
    "cve_language":    "language",
    "cve_description": "cve_desc_raw",
    "cwe_description": "cwe_desc_raw",
})

print(f"\nCWE 分布 Top-15:")
print(df["cwe_id"].value_counts().head(15))
df[["cve_id", "cwe_id", "language"]].head()

过滤无真实 CWE 的样本:     3725 条 (移除 59)
按 CWE explode 后:          4721 条

CWE 分布 Top-15:
cwe_id
CWE-125    557
CWE-119    541
CWE-787    435
CWE-190    305
CWE-416    222
CWE-476    215
CWE-20     186
CWE-200    165
CWE-399    151
CWE-362    145
CWE-264    122
CWE-120    107
CWE-400    105
CWE-189     99
CWE-401     72
Name: count, dtype: int64


,cve_id,cwe_id,language
0,CVE-2022-24754,CWE-787,C
1,CVE-2022-24754,CWE-120,C
2,CVE-2022-24754,CWE-1284,C
3,CVE-2023-23457,CWE-119,C++
4,CVE-2012-2319,CWE-264,C


In [6]:
# ── Cell 5: 同源 CVE 合并（可选） ──
# 当 MERGE_FUNCS_AS_CVE=True 时，按 (cve_id, commit_id, cwe_id) 将同一 CVE
# 涉及到的多个漏洞函数的代码用分隔符拼接为单条记录。

SEP = "\n\n/* ---- next function ---- */\n\n"

if MERGE_FUNCS_AS_CVE:
    print("启用同源 CVE 合并 (MERGE_FUNCS_AS_CVE=True)...")
    merged_rows = []
    for (cve_id, commit_id, cwe_id), grp in df.groupby(
        ["cve_id", "commit_id", "cwe_id"], sort=False
    ):
        first = grp.iloc[0].to_dict()
        first["cve_id"]    = cve_id
        first["commit_id"] = commit_id
        first["cwe_id"]    = cwe_id
        first["vul_code"]   = SEP.join(grp["vul_code"].tolist())
        first["fixed_code"] = SEP.join(grp["fixed_code"].tolist())
        merged_rows.append(first)

    df = pd.DataFrame(merged_rows).reset_index(drop=True)

    # 合并后再次过滤超长代码
    df = df[
        df["vul_code"].str.len().between(MIN_CODE_LEN, MAX_CODE_LEN, inclusive="both")
        & df["fixed_code"].str.len().between(MIN_CODE_LEN, MAX_CODE_LEN, inclusive="both")
    ].reset_index(drop=True)

    print(f"合并后样本数: {len(df)}")
else:
    print(f"跳过同源 CVE 合并 (MERGE_FUNCS_AS_CVE=False)，当前样本数: {len(df)}")

df[["cve_id", "cwe_id", "language"]].head()

启用同源 CVE 合并 (MERGE_FUNCS_AS_CVE=True)...
合并后样本数: 2531


,cve_id,cwe_id,language
0,CVE-2022-24754,CWE-787,C
1,CVE-2022-24754,CWE-120,C
2,CVE-2022-24754,CWE-1284,C
3,CVE-2023-23457,CWE-119,C++
4,CVE-2012-2319,CWE-264,C


In [7]:
# ── Cell 6: 代码格式化 (clang-format) ──
print("格式化 vul_code ...")
vul_results = auto_format_c_code_batch(
    df["vul_code"].tolist(),
    style=ClangFormatStyle.LLVM,
    on_error="keep",
    show_progress=True,
)
df["vul_code"] = [r[0] for r in vul_results]

print("格式化 fixed_code ...")
fixed_results = auto_format_c_code_batch(
    df["fixed_code"].tolist(),
    style=ClangFormatStyle.LLVM,
    on_error="keep",
    show_progress=True,
)
df["fixed_code"] = [r[0] for r in fixed_results]

low_conf_vul   = sum(1 for r in vul_results   if r[1].confidence < 0.6)
low_conf_fixed = sum(1 for r in fixed_results if r[1].confidence < 0.6)
print(f"\nvul_code   低置信度样本数 (<0.6): {low_conf_vul}")
print(f"fixed_code 低置信度样本数 (<0.6): {low_conf_fixed}")
print(f"格式化完成，总样本数: {len(df)}")

# ── 格式化后再次过滤 vul_code == fixed_code ──
# clang-format 会消除仅有空白/缩进差异的代码对，必须重新过滤。
before_dedup = len(df)
df = df[df["vul_code"] != df["fixed_code"]].reset_index(drop=True)
removed = before_dedup - len(df)
print(f"\n格式化后 vul!=fixed: 移除 {removed} 条，剩余 {len(df)} 条")

格式化 vul_code ...


Formatting: 100%|██████████| 2531/2531 [00:19<00:00, 127.13snippet/s]

格式化 fixed_code ...



Formatting: 100%|██████████| 2531/2531 [00:20<00:00, 124.93snippet/s]


vul_code   低置信度样本数 (<0.6): 82
fixed_code 低置信度样本数 (<0.6): 78
格式化完成，总样本数: 2531

格式化后 vul!=fixed: 移除 4 条，剩余 2527 条


In [8]:
# ── Cell 7: 填充 cwe_desc ──
# 经过 Cell 4 后 cwe_id 已是单个字符串（如 "CWE-119"），统一以 CWEUtils 为权威源。

cwe_utils = CWEUtils(PROJECT_ROOT / "data/raw/cwe/cwec_v4.19.1.xml")


def get_cwe_desc(cwe_id):
    if not isinstance(cwe_id, str):
        return None
    cid = cwe_id.strip()
    if not cid:
        return None
    prompt = cwe_utils.get_prompt(cid)
    if "not found in the CWE dataset" in prompt:
        return None
    return prompt


df["cwe_desc"] = df["cwe_id"].apply(get_cwe_desc)

filled = df["cwe_desc"].notna().sum()
print(f"总样本数:              {len(df)}")
print(f"cwe_desc 已填充:       {filled}")
print(f"cwe_desc 为空:         {len(df) - filled}")
print(f"cwe_desc 填充率:       {filled / len(df) * 100:.2f}%")
df[["cwe_id", "cwe_desc"]].head()

总样本数:              2527
cwe_desc 已填充:       2527
cwe_desc 为空:         0
cwe_desc 填充率:       100.00%


,cwe_id,cwe_desc
0,CWE-787,- CWE ID: CWE-787\n- Name: Out-of-bounds Write...
1,CWE-120,- CWE ID: CWE-120\n- Name: Buffer Copy without...
2,CWE-1284,- CWE ID: CWE-1284\n- Name: Improper Validatio...
3,CWE-119,- CWE ID: CWE-119\n- Name: Improper Restrictio...
4,CWE-264,"- CWE ID: CWE-264\n- Name: Permissions, Privil..."


In [9]:
# ── Cell 8: 填充 cve_desc ──
# 策略：优先使用 ReposVul 自带的 cve_description 字段，缺失时回退 CVEUtils。

cve_utils = CVEUtils(PROJECT_ROOT / "data/raw/cwe/CVE_v2026.03.18.json")


def _normalize_cve_desc_raw(val):
    if val is None or not isinstance(val, str):
        return None
    s = val.strip()
    return s if s and s.lower() != "none" else None


def get_cve_desc(row):
    raw_desc = _normalize_cve_desc_raw(row.get("cve_desc_raw"))
    if raw_desc:
        return raw_desc

    cve_id = row.get("cve_id")
    if not cve_id or not str(cve_id).strip():
        return None
    prompt = cve_utils.get_prompt(str(cve_id).strip())
    if "was not found in the CVE dataset" in prompt:
        return None
    return prompt


df["cve_desc"] = df.apply(get_cve_desc, axis=1)

filled   = df["cve_desc"].notna().sum()
from_raw = df["cve_desc_raw"].apply(_normalize_cve_desc_raw).notna().sum()
print(f"总样本数:                  {len(df)}")
print(f"cve_desc 已填充:           {filled}")
print(f"  └─ 来自原始字段:         {from_raw}")
print(f"  └─ 来自 CVEUtils 补充:   {filled - from_raw}")
print(f"cve_desc 为空:             {len(df) - filled}")
print(f"cve_desc 填充率:           {filled / len(df) * 100:.2f}%")
df[["cve_id", "cve_desc"]].head()

总样本数:                  2527
cve_desc 已填充:           2527
  └─ 来自原始字段:         2527
  └─ 来自 CVEUtils 补充:   0
cve_desc 为空:             0
cve_desc 填充率:           100.00%


,cve_id,cve_desc
0,CVE-2022-24754,PJSIP is a free and open source multimedia com...
1,CVE-2022-24754,PJSIP is a free and open source multimedia com...
2,CVE-2022-24754,PJSIP is a free and open source multimedia com...
3,CVE-2023-23457,A Segmentation fault was found in UPX in PackL...
4,CVE-2012-2319,Multiple buffer overflows in the hfsplus files...


In [10]:
# ── Cell 9: 生成 diff 字段 ──
def compute_diff(vul_code: str, fixed_code: str) -> str:
    """使用 difflib.unified_diff 计算函数级代码变更差异。"""
    a = vul_code.splitlines(keepends=True)
    b = fixed_code.splitlines(keepends=True)
    diff_lines = list(difflib.unified_diff(a, b, fromfile="vul_code", tofile="fixed_code"))
    return "".join(diff_lines) if diff_lines else ""


df["diff"] = df.apply(lambda r: compute_diff(r["vul_code"], r["fixed_code"]), axis=1)

empty_diff = (df["diff"] == "").sum()
print(f"总样本数:         {len(df)}")
print(f"diff 非空:        {len(df) - empty_diff}")
print(f"diff 为空:        {empty_diff}  （vul_code 与 fixed_code 完全一致，理论上已在前置步骤过滤）")
df[["vul_code", "fixed_code", "diff"]].head(1)

总样本数:         2527
diff 非空:        2527
diff 为空:        0  （vul_code 与 fixed_code 完全一致，理论上已在前置步骤过滤）


,vul_code,fixed_code,diff
0,static pj_status_t\nrespond_digest(pj_pool_t *...,static pj_status_t\nrespond_digest(pj_pool_t *...,"--- vul_code\n+++ fixed_code\n@@ -5,6 +5,7 @@\..."


In [11]:
# ── Cell 10: 分配主键与 source 字段 ──
df = df.reset_index(drop=True)
df["id"]     = df.index
df["source"] = "ReposVul"

print(f"id 范围:   0 ~ {df['id'].max()}")
print(f"source:    {df['source'].unique().tolist()}")
print(f"总样本数:  {len(df)}")
df[["id", "source", "cwe_id", "cve_id", "language"]].head()

id 范围:   0 ~ 2526
source:    ['ReposVul']
总样本数:  2527


,id,source,cwe_id,cve_id,language
0,0,ReposVul,CWE-787,CVE-2022-24754,C
1,1,ReposVul,CWE-120,CVE-2022-24754,C
2,2,ReposVul,CWE-1284,CVE-2022-24754,C
3,3,ReposVul,CWE-119,CVE-2023-23457,C++
4,4,ReposVul,CWE-264,CVE-2012-2319,C


In [12]:
# ── Cell 11: 导出 jsonl 文件 ──
FINAL_COLS = [
    "id", "source", "language",
    "cwe_id", "cve_id",
    "cwe_desc", "cve_desc",
    "vul_code", "fixed_code",
    "diff",
]

df_out = df[FINAL_COLS].copy()

OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
df_out.to_json(
    OUTPUT_PATH,
    orient="records",
    lines=True,
    force_ascii=False,
)

print(f"导出完成: {OUTPUT_PATH}")
print(f"总样本数: {len(df_out)}")
print(f"\n字段统计：")
for col in FINAL_COLS:
    null_count = df_out[col].isna().sum()
    fill_rate  = (1 - null_count / len(df_out)) * 100
    print(f"  {col:<14}: 填充率 {fill_rate:6.2f}%  (空值 {null_count} 条)")

df_out.head()

导出完成: <PROJECT_ROOT>/data/raw/vul/ReposVul_Merged.jsonl
总样本数: 2527

字段统计：
  id            : 填充率 100.00%  (空值 0 条)
  source        : 填充率 100.00%  (空值 0 条)
  language      : 填充率 100.00%  (空值 0 条)
  cwe_id        : 填充率 100.00%  (空值 0 条)
  cve_id        : 填充率 100.00%  (空值 0 条)
  cwe_desc      : 填充率 100.00%  (空值 0 条)
  cve_desc      : 填充率 100.00%  (空值 0 条)
  vul_code      : 填充率 100.00%  (空值 0 条)
  fixed_code    : 填充率 100.00%  (空值 0 条)
  diff          : 填充率 100.00%  (空值 0 条)


,id,source,language,cwe_id,cve_id,cwe_desc,cve_desc,vul_code,fixed_code,diff
0,0,ReposVul,C,CWE-787,CVE-2022-24754,- CWE ID: CWE-787\n- Name: Out-of-bounds Write...,PJSIP is a free and open source multimedia com...,static pj_status_t\nrespond_digest(pj_pool_t *...,static pj_status_t\nrespond_digest(pj_pool_t *...,"--- vul_code\n+++ fixed_code\n@@ -5,6 +5,7 @@\..."
1,1,ReposVul,C,CWE-120,CVE-2022-24754,- CWE ID: CWE-120\n- Name: Buffer Copy without...,PJSIP is a free and open source multimedia com...,static pj_status_t\nrespond_digest(pj_pool_t *...,static pj_status_t\nrespond_digest(pj_pool_t *...,"--- vul_code\n+++ fixed_code\n@@ -5,6 +5,7 @@\..."
2,2,ReposVul,C,CWE-1284,CVE-2022-24754,- CWE ID: CWE-1284\n- Name: Improper Validatio...,PJSIP is a free and open source multimedia com...,static pj_status_t\nrespond_digest(pj_pool_t *...,static pj_status_t\nrespond_digest(pj_pool_t *...,"--- vul_code\n+++ fixed_code\n@@ -5,6 +5,7 @@\..."
3,3,ReposVul,C++,CWE-119,CVE-2023-23457,- CWE ID: CWE-119\n- Name: Improper Restrictio...,A Segmentation fault was found in UPX in PackL...,void PackLinuxElf64::PackLinuxElf64help1(Input...,void PackLinuxElf64::PackLinuxElf64help1(Input...,"--- vul_code\n+++ fixed_code\n@@ -3,7 +3,8 @@\..."
4,4,ReposVul,C,CWE-264,CVE-2012-2319,"- CWE ID: CWE-264\n- Name: Permissions, Privil...",Multiple buffer overflows in the hfsplus files...,"int hfsplus_rename_cat(u32 cnid, struct inode ...","int hfsplus_rename_cat(u32 cnid, struct inode ...","--- vul_code\n+++ fixed_code\n@@ -18,6 +18,10 ..."
